# Lab-2d: Data Preparation with EMR on EC2

**Persona:** Data Engineer &nbsp;|&nbsp; **Lab:** 2 - Data Prep

## Overview

This notebook produces the same feature-ready **train** and **test** datasets as Lab 2B, but runs the preprocessing on **EMR on EC2** with Apache Spark instead of a SageMaker Processing Job.

You'll use the **bank marketing dataset** (UCI Machine Learning Repository) to predict whether a customer will subscribe to a term deposit. The preprocessing steps are identical to 2B:

1. Download the raw dataset and stage it in Amazon S3
2. Run a PySpark job on EMR on EC2 that:
   - Label-encodes categorical features
   - Encodes the binary target (`y`)
   - Performs a stratified train/test split
   - Writes `train.csv` and `test.csv` back to S3
3. Record the S3 output locations used downstream by Lab 3A

### Why EMR on EC2?

**EMR on EC2** gives you a managed Hadoop/Spark cluster on instances you choose. You control instance types, node counts, bootstrap actions, and the software stack; EMR handles provisioning, configuration, and monitoring.

Choose it over EMR Serverless when you need:
- **Cluster-level control** — custom AMIs, bootstrap actions, system packages, SSH access
- **Long-running or repeated jobs** — a persistent cluster amortizes the ~7-10 minute startup across many steps
- **The wider EMR ecosystem** — Hive, HBase, Presto, Flink, or notebooks attached to the same cluster
- **Cost control at scale** — Spot instances for task nodes, Reserved/Savings Plans for core nodes

We run a **transient cluster**: it provisions, runs one step, and terminates itself. That pattern gives you EMR on EC2's control without paying for an idle cluster.

### Output contract (consumed by Lab 3A)

- `s3://<default-bucket>/bank-marketing-lab/data/train/train.csv`
- `s3://<default-bucket>/bank-marketing-lab/data/test/test.csv`

Each CSV has the **target as the first column** and **no header row**, matching what the XGBoost training script in Lab 3A expects.

<div style="padding: 15px; background-color: #fff3cd; border-left: 5px solid #ffc107; color: #856404;">
<strong>⚠️ Important:</strong> Labs 2B, 2C, 2D, and 2E all write to the <strong>same S3 output paths</strong>. Whichever you run last is what Lab 3A will consume. Pick one option per run-through, or change the <code>prefix</code> variable below if you want to compare outputs side by side.
</div>

---

## Section 1: Setup

Install the SageMaker Python SDK v3 and supporting libraries, then initialize the session. The first cell may take 1-2 minutes and will restart the kernel.

In [ ]:
# Install required packages
# This may take 1-2 minutes on first run. Ignore dependency conflict warnings.
!pip install --upgrade pip -q

# Clean uninstall to avoid cached version conflicts
%pip uninstall -y sagemaker sagemaker-core sagemaker-train sagemaker-serve sagemaker-mlops -q

# Reinstall with compatible versions
%pip install --no-cache-dir "sagemaker>=3.17,<4" \
    "pandas" "boto3>=1.34" -q

# Restart kernel to pick up updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

In [ ]:
# Import required libraries
import boto3
import sagemaker
import pandas as pd
import json
import os
import io
import time
import importlib.metadata

# SageMaker v3 session helpers
from sagemaker.core.helper.session_helper import Session, get_execution_role

print('✓ All libraries imported successfully')
print(f'SageMaker SDK version: {importlib.metadata.version("sagemaker")}')
print(f'boto3 version: {importlib.metadata.version("boto3")}')

In [ ]:
# Initialize SageMaker session and get AWS configuration
import os, sys
sagemaker_session = Session()
region = sagemaker_session.boto_region_name
role = get_execution_role()
bucket = sagemaker_session.default_bucket()

# Data prefix from the single shared source (repo-root .env via shared loader).
for _c in [os.getcwd(), *[str(p) for p in __import__('pathlib').Path(os.getcwd()).parents]]:
    if os.path.exists(os.path.join(_c, 'workshop_env.py')):
        sys.path.insert(0, _c); break
from workshop_env import load_workshop_env
prefix = load_workshop_env()['DATA_PREFIX']

account_id = boto3.client('sts', region_name=region).get_caller_identity()['Account']

print('AWS Configuration:')
print(f'  Region: {region}')
print(f'  Account: {account_id}')
print(f'  S3 Bucket: {bucket}')
print(f'  IAM Role: {role}')
print(f'  Data Prefix: {prefix}')
print('\n✓ SageMaker session initialized successfully')

---

## Section 2: Acquire Raw Dataset and Stage in S3

Spark reads its input from Amazon S3. We download the raw bank marketing dataset locally, then upload the raw CSV so the Spark job can read it directly from S3.

### Dataset Features

- **Client Demographics**: age, job type, marital status, education level
- **Financial Profile**: credit default status, housing loans, personal loans
- **Campaign Details**: contact method, timing (month/day), call duration, number of contacts, previous outcomes
- **Economic Indicators**: employment variation rate, consumer price/confidence indices, Euribor rate, employment numbers
- **Target Variable**: binary - did the client subscribe to a term deposit (yes/no)

In [ ]:
# Download and extract the dataset
print('Downloading bank marketing dataset...')
!wget -N https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip
!unzip -o bank-additional.zip
print('\n✓ Dataset downloaded and extracted')

In [ ]:
# Quick look at the raw data (semicolon-separated)
raw_df = pd.read_csv('bank-additional/bank-additional-full.csv', sep=';')
print(f'Raw shape: {raw_df.shape}')
raw_df.head()

In [ ]:
# Upload the raw dataset to S3 as the Spark job input
raw_data_s3 = sagemaker_session.upload_data(
    'bank-additional/bank-additional-full.csv',
    bucket,
    f'{prefix}/data/raw'
)
print(f'Raw data staged at: {raw_data_s3}')

---

## Section 3: Author the PySpark Job

This is the same preprocessing logic as Lab 2B, expressed in Spark. Read it carefully — two details determine whether the output actually matches what 2B produced.

### Detail 1: `stringOrderType="alphabetAsc"` is required

scikit-learn's `LabelEncoder` sorts distinct values **alphabetically** and assigns `0..n-1`. Spark's `StringIndexer` defaults to `frequencyDesc` — the most frequent value gets `0`.

| Category | sklearn `LabelEncoder` (2B) | Spark default `frequencyDesc` | Spark `alphabetAsc` |
|---|---:|---:|---:|
| `admin.` | 0 | 1 | 0 |
| `blue-collar` | 1 | 0 | 1 |
| `management` | 2 | 2 | 2 |
| `retired` | 3 | 5 | 3 |

Nothing errors if you leave the default. You simply get a **silently different dataset** than 2B, and any model comparison across labs becomes meaningless.

### Detail 2: cast the indexed columns to `int`

`StringIndexer` emits `DoubleType`, so the CSV would contain `3.0` where 2B wrote `3`. XGBoost reads both, but casting keeps the two labs byte-comparable.

### On the train/test split

`sampleBy` draws each class independently at the same rate — Spark's equivalent of sklearn's `stratify=y`. **Row membership will not match 2B** because the RNGs differ. The class balance, split ratio, schema, and column order do match, and that is what the downstream contract depends on.

In [ ]:
# Create the PySpark job that runs on the cluster
os.makedirs('spark', exist_ok=True)

spark_script = '''import argparse

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--input-uri", required=True)
    p.add_argument("--train-output-uri", required=True)
    p.add_argument("--test-output-uri", required=True)
    p.add_argument("--test-size", type=float, default=0.2)
    p.add_argument("--random-state", type=int, default=42)
    return p.parse_args()


def main():
    args = parse_args()
    spark = SparkSession.builder.appName("bank-marketing-prep").getOrCreate()

    # ---- Load ------------------------------------------------------------
    # The raw UCI file is semicolon-separated with a header.
    df = (spark.read
          .option("header", True)
          .option("sep", ";")
          .option("inferSchema", True)
          .csv(args.input_uri))
    print(f"Loaded raw data: {df.count()} rows x {len(df.columns)} cols")

    # Sanitize column names: the raw UCI file uses dotted names
    # (emp.var.rate, cons.price.idx, cons.conf.idx, nr.employed). Spark SQL
    # treats a dot as struct-field navigation, so referencing them by name
    # fails with UNRESOLVED_COLUMN. Replace '.' with '_' so downstream code
    # (and the Iceberg schema in 5-data-prep.yaml) uses emp_var_rate etc.
    for old_name in df.columns:
        new_name = old_name.replace(".", "_")
        if new_name != old_name:
            df = df.withColumnRenamed(old_name, new_name)

    original_cols = df.columns
    cat_cols = [f.name for f in df.schema.fields
                if isinstance(f.dataType, StringType) and f.name != "y"]
    print(f"Categorical columns to encode: {cat_cols}")

    # ---- Encode categoricals --------------------------------------------
    # stringOrderType="alphabetAsc" is REQUIRED for parity with Lab 2B.
    #
    # scikit-learn's LabelEncoder sorts the distinct values alphabetically and
    # assigns 0..n-1. Spark's StringIndexer defaults to "frequencyDesc" (most
    # frequent value gets 0), which produces a DIFFERENT integer for the same
    # category. Nothing errors -- you just get a silently different dataset
    # than 2B produced, and any model comparison across labs becomes invalid.
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}__idx",
                      stringOrderType="alphabetAsc", handleInvalid="keep")
        for c in cat_cols
    ]
    df = Pipeline(stages=indexers).fit(df).transform(df)

    # StringIndexer emits DoubleType. Cast to int so the CSV contains "3" and
    # not "3.0" -- pandas/XGBoost read both, but 2B wrote ints and we want the
    # two labs to produce byte-comparable files.
    for c in cat_cols:
        df = df.drop(c).withColumnRenamed(f"{c}__idx", c)
        df = df.withColumn(c, F.col(c).cast("int"))

    # ---- Encode target ---------------------------------------------------
    df = df.withColumn("y", F.when(F.col("y") == "yes", 1).otherwise(0).cast("int"))

    # StringIndexer appended its columns at the end, so restore 2B's column order.
    df = df.select(*original_cols)

    # ---- Stratified split ------------------------------------------------
    # sampleBy draws each class independently at the same rate, which is the
    # Spark equivalent of sklearn's stratify=y. Row membership will NOT match
    # 2B (different RNG), but the class balance and split ratio do -- and that
    # is what the downstream contract actually depends on.
    train_frac = 1.0 - args.test_size
    df = df.withColumn("_rid", F.monotonically_increasing_id())
    train = df.sampleBy("y", fractions={0: train_frac, 1: train_frac},
                        seed=args.random_state)
    # left_anti keeps every row not sampled into train, without the dedup
    # side-effect that subtract()/exceptAll() would introduce.
    test = df.join(train.select("_rid"), on="_rid", how="left_anti")

    train = train.drop("_rid")
    test = test.drop("_rid")

    # ---- Write -----------------------------------------------------------
    # Target first, no header: the CSV convention the XGBoost training script
    # in Lab 3A expects.
    feature_cols = [c for c in original_cols if c != "y"]

    for name, sdf, uri in (("train", train, args.train_output_uri),
                           ("test", test, args.test_output_uri)):
        n = sdf.count()
        pos = sdf.filter(F.col("y") == 1).count()
        print(f"{name}: {n} rows, positive rate {pos / n:.4f}")
        # coalesce(1) so the stage emits a single part file, which the notebook
        # then renames to train.csv / test.csv. At 41K rows this is trivial; on
        # a genuinely large dataset you would leave it partitioned and teach the
        # consumer to read a prefix instead.
        (sdf.select("y", *feature_cols)
            .coalesce(1)
            .write.mode("overwrite")
            .option("header", False)
            .csv(uri))
        print(f"{name}: written to {uri}")

    print("Preprocessing complete")
    spark.stop()


if __name__ == "__main__":
    main()
'''

with open('spark/preprocessing_spark.py', 'w') as f:
    f.write(spark_script)

print('✓ PySpark job created at spark/preprocessing_spark.py')

In [ ]:
# Upload the job script to S3 so the cluster can fetch it
script_s3 = sagemaker_session.upload_data(
    'spark/preprocessing_spark.py',
    bucket,
    f'{prefix}/scripts'
)
print(f'Spark script staged at: {script_s3}')

---

## Section 4: Provision the Cluster and Run the Job

EMR on EC2 needs two service roles and a subnet. We resolve all three before launching.

### EMR service roles (pre-provisioned)

These two roles are created by the `5-data-prep.yaml` CloudFormation stack — you do not create them here:

- **`<ProjectName>-emr-service-role`** — EMR itself uses this to provision EC2 instances on your behalf
- **`<ProjectName>-emr-ec2-instance-role`** — the instance profile the cluster nodes run under; this is what reaches S3

The cell below verifies they exist. If they're missing, deploy the `5-data-prep.yaml` stack (and confirm `PROJECT_NAME` matches the deploy-time `ProjectName`).

In [ ]:
emr = boto3.client('emr', region_name=region)
ec2 = boto3.client('ec2', region_name=region)

# EMR service role + EC2 instance profile are pre-provisioned by the
# 5-data-prep.yaml stack. Match PROJECT_NAME to the deploy-time ProjectName.
# We reference them by name only. The notebook's SageMaker execution role does
# not have IAM read permissions (iam:GetRole / iam:GetInstanceProfile), so we
# do NOT pre-check them here. run_job_flow below references these names and
# fails with a clear error if either resource is missing.
PROJECT_NAME = os.environ.get('PROJECT_NAME', 'bank-marketing-prediction')
SERVICE_ROLE = f'{PROJECT_NAME}-emr-service-role'
INSTANCE_PROFILE = f'{PROJECT_NAME}-emr-ec2-instance-role'

print(f'✓ EMR service role:     {SERVICE_ROLE}')
print(f'✓ EMR EC2 instance profile: {INSTANCE_PROFILE}')
print('  (pre-provisioned by 5-data-prep.yaml; referenced by run_job_flow below)')

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #0c5460; color: #0c5460;">
<strong>ℹ️ Note:</strong> The default <code>EMR_EC2_DefaultRole</code> includes broad S3 access. In a production account you would replace it with a role scoped to only the buckets the job needs — the same principle applied to the EMR Serverless runtime role in Lab 2C.
</div>

In [ ]:
# Pick a subnet for the cluster. EMR on EC2 requires one (unlike EMR Serverless).
# Prefer a subnet in the SageMaker domain's VPC so the cluster sits alongside the
# rest of the workshop infrastructure.
subnets = ec2.describe_subnets()['Subnets']
if not subnets:
    raise RuntimeError('No subnets found. Specify subnet_id manually below.')

# Prefer a private subnet if one is tagged as such; otherwise take the first available.
private = [s for s in subnets
           if any(t.get('Key') == 'Name' and 'rivate' in t.get('Value', '')
                  for t in s.get('Tags', []))]
chosen = (private or subnets)[0]
subnet_id = chosen['SubnetId']

print(f'Using subnet: {subnet_id}')
print(f'  VPC:  {chosen["VpcId"]}')
print(f'  AZ:   {chosen["AvailabilityZone"]}')
print(f'  Type: {"private (tagged)" if private else "first available"}')
print('\nOverride by setting subnet_id manually if your VPC requires a specific subnet.')

### Launch a transient cluster

The configuration below launches, runs one step, and self-terminates.

Key settings:

- **`KeepJobFlowAliveWhenNoSteps=False`** — this is what makes the cluster transient. Set it `True` and the cluster stays up after the step finishes, waiting for more work (and billing you).
- **`ActionOnFailure='TERMINATE_CLUSTER'`** — a failed step shuts the cluster down rather than leaving it idle.
- **`--deploy-mode cluster`** — the Spark driver runs on the cluster, not on the submitting client.
- **`command-runner.jar`** — EMR's standard mechanism for running a shell command (here, `spark-submit`) as a step.

In [ ]:
staging_train = f's3://{bucket}/{prefix}/data/spark-staging/train'
staging_test = f's3://{bucket}/{prefix}/data/spark-staging/test'
logs_uri = f's3://{bucket}/{prefix}/emr-ec2-logs'

RELEASE_LABEL = 'emr-7.5.0'

cluster = emr.run_job_flow(
    Name='bank-marketing-data-prep',
    LogUri=logs_uri,
    ReleaseLabel=RELEASE_LABEL,
    Applications=[{'Name': 'Spark'}],
    Instances={
        'InstanceGroups': [
            {
                'Name': 'Primary',
                'InstanceRole': 'MASTER',
                'InstanceType': 'm5.xlarge',
                'InstanceCount': 1,
            },
            {
                'Name': 'Core',
                'InstanceRole': 'CORE',
                'InstanceType': 'm5.xlarge',
                'InstanceCount': 2,
            },
        ],
        'Ec2SubnetId': subnet_id,
        # The transient-cluster switch: terminate once the step queue drains.
        'KeepJobFlowAliveWhenNoSteps': False,
        'TerminationProtected': False,
    },
    Steps=[
        {
            'Name': 'bank-marketing-preprocessing',
            # A failed step should not leave an idle cluster billing.
            'ActionOnFailure': 'TERMINATE_CLUSTER',
            'HadoopJarStep': {
                'Jar': 'command-runner.jar',
                'Args': [
                    'spark-submit',
                    '--deploy-mode', 'cluster',
                    '--conf', 'spark.executor.cores=2',
                    '--conf', 'spark.executor.memory=4g',
                    '--conf', 'spark.executor.instances=2',
                    script_s3,
                    '--input-uri', raw_data_s3,
                    '--train-output-uri', staging_train,
                    '--test-output-uri', staging_test,
                    '--test-size', '0.2',
                    '--random-state', '42',
                ],
            },
        }
    ],
    JobFlowRole=INSTANCE_PROFILE,
    ServiceRole=SERVICE_ROLE,
    VisibleToAllUsers=True,
    Tags=[
        {'Key': 'Project', 'Value': 'genai-ml-standardization'},
        {'Key': 'Lab', 'Value': 'lab2d-emr-ec2'},
    ],
)

cluster_id = cluster['JobFlowId']
print(f'✓ Cluster launched: {cluster_id}')
print(f'  Release: {RELEASE_LABEL}')
print(f'  Nodes:   1 primary + 2 core (m5.xlarge)')
print(f'  Logs:    {logs_uri}')
print('\nProvisioning takes ~7-10 minutes before the step starts.')

In [ ]:
# Poll the cluster until it terminates (transient cluster: one step, then done)
TERMINAL = {'TERMINATED', 'TERMINATED_WITH_ERRORS'}
start = time.time()
last = None

while True:
    desc = emr.describe_cluster(ClusterId=cluster_id)['Cluster']
    state = desc['Status']['State']
    msg = desc['Status'].get('StateChangeReason', {}).get('Message', '')

    if state != last:
        print(f'  [{int(time.time() - start):>4}s] {state}  {msg}', flush=True)
        last = state

    if state in TERMINAL:
        break
    time.sleep(30)

elapsed = int(time.time() - start)
steps = emr.list_steps(ClusterId=cluster_id)['Steps']
step_state = steps[0]['Status']['State'] if steps else 'UNKNOWN'

print(f'\nCluster state: {state}  (after {elapsed}s)')
print(f'Step state:    {step_state}')

if state == 'TERMINATED' and step_state == 'COMPLETED':
    print('\n✓ Preprocessing step completed and cluster self-terminated')
else:
    print(f'\n✗ Run did not complete cleanly')
    if steps:
        fd = steps[0]['Status'].get('FailureDetails', {})
        print(f'  Reason: {fd.get("Reason", "(none)")}')
        print(f'  Log:    {fd.get("LogFile", logs_uri)}')
    print(f'\n  Full logs: {logs_uri}/{cluster_id}/')
    raise RuntimeError(
        f'EMR cluster {cluster_id} finished in state {state} with step state '
        f'{step_state}. Stopping here so Section 5 does not mask the cause '
        f'with a "No Spark output found" error.'
    )

---

## Section 5: Consolidate Spark Output into `train.csv` / `test.csv`

Spark writes a directory of part files (`part-00000-....csv`), plus `_SUCCESS` markers. Lab 3A expects a single object at an exact key: `.../data/train/train.csv`.

Because we called `coalesce(1)` in the job, each output directory contains exactly one part file. We copy it to the final key and clean up the staging prefix.

<div style="padding: 15px; background-color: #d1ecf1; border-left: 5px solid #0c5460; color: #0c5460;">
<strong>ℹ️ Note:</strong> On a genuinely large dataset you would <strong>not</strong> coalesce to a single file — you would leave the output partitioned and have the consumer read the whole prefix. We consolidate here only because Lab 3A's contract specifies a single named object.
</div>

In [ ]:
# Promote the single Spark part file to the exact key Lab 3A reads
s3 = boto3.client('s3', region_name=region)


def promote_part_file(staging_prefix, final_key):
    """Copy the single coalesced part file to its final name, then remove staging."""
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=staging_prefix)
    if 'Contents' not in resp:
        raise RuntimeError(f'No Spark output found under s3://{bucket}/{staging_prefix}')

    parts = [o['Key'] for o in resp['Contents']
             if o['Key'].endswith('.csv') and '/part-' in o['Key']]
    if len(parts) != 1:
        raise RuntimeError(f'Expected exactly 1 part file, found {len(parts)}: {parts}')

    s3.copy_object(
        Bucket=bucket,
        CopySource={'Bucket': bucket, 'Key': parts[0]},
        Key=final_key
    )
    size_mb = s3.head_object(Bucket=bucket, Key=final_key)['ContentLength'] / 1e6
    print(f'  s3://{bucket}/{final_key}  ({size_mb:.2f} MB)')

    # Remove the staging directory (part file, _SUCCESS markers, committer files)
    s3.delete_objects(
        Bucket=bucket,
        Delete={'Objects': [{'Key': o['Key']} for o in resp['Contents']]}
    )


print('Consolidating Spark output:')
promote_part_file(f'{prefix}/data/spark-staging/train/', f'{prefix}/data/train/train.csv')
promote_part_file(f'{prefix}/data/spark-staging/test/', f'{prefix}/data/test/test.csv')
print('\n✓ Outputs promoted to the Lab 3A contract paths')

---

## Section 6: Verify Outputs

Confirm the train and test datasets landed in S3 and preview the first few rows. These are the exact paths Lab 3A (`lab3a_traditional_ml_experimenation.ipynb`) uses for training and evaluation.

In [ ]:
# Final S3 locations of the prepared datasets
train_s3 = f's3://{bucket}/{prefix}/data/train/train.csv'
test_s3 = f's3://{bucket}/{prefix}/data/test/test.csv'

print('Prepared datasets (used by Lab 3A):')
print(f'  Train: {train_s3}')
print(f'  Test:  {test_s3}')

In [ ]:
# Preview the prepared train dataset directly from S3
def preview_s3_csv(s3_uri, n=5):
    _, _, rest = s3_uri.partition('s3://')
    b, _, key = rest.partition('/')
    obj = boto3.client('s3', region_name=region).get_object(Bucket=b, Key=key)
    df = pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)
    print(f'{s3_uri}  ->  shape {df.shape} (col 0 is the target)')
    return df.head(n)


preview_s3_csv(train_s3)

In [ ]:
# Preview the prepared test dataset
preview_s3_csv(test_s3)

In [ ]:
# Sanity-check the contract: stratification held, and the split ratio is right
def load_s3_csv(s3_uri):
    _, _, rest = s3_uri.partition('s3://')
    b, _, key = rest.partition('/')
    obj = boto3.client('s3', region_name=region).get_object(Bucket=b, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()), header=None)


tr, te = load_s3_csv(train_s3), load_s3_csv(test_s3)
total = len(tr) + len(te)

print(f'Train rows: {len(tr):>6}  ({len(tr)/total:.1%})')
print(f'Test rows:  {len(te):>6}  ({len(te)/total:.1%})')
print(f'Total:      {total:>6}   (raw dataset had {len(raw_df)})')
print()
print(f'Train positive rate: {tr[0].mean():.4f}')
print(f'Test positive rate:  {te[0].mean():.4f}')
print(f'Raw positive rate:   {(raw_df["y"] == "yes").mean():.4f}')
print()
print(f'Columns: {tr.shape[1]} (1 target + {tr.shape[1]-1} features)')

assert tr.shape[1] == te.shape[1] == 21, 'Expected 21 columns (target + 20 features)'
assert abs(tr[0].mean() - te[0].mean()) < 0.01, 'Stratification looks off'
print('\n✓ Output contract verified')

---

## Section 7: Clean Up

Because we set `KeepJobFlowAliveWhenNoSteps=False`, the cluster terminated itself after the step finished — there is no idle compute to clean up.

The cell below confirms that and shows how to terminate manually if you ever launch a persistent cluster.

The EMR **service role** and **EC2 instance profile** used by the cluster are provisioned and owned by the `5-data-prep.yaml` CloudFormation stack. Do **not** delete them here — they are shared across runs and removed when that stack is torn down.

In [ ]:
# Confirm the cluster is gone
state = emr.describe_cluster(ClusterId=cluster_id)['Cluster']['Status']['State']
print(f'Cluster {cluster_id}: {state}')

if state in ('TERMINATED', 'TERMINATED_WITH_ERRORS'):
    print('✓ No running compute — the transient cluster self-terminated as configured')
else:
    print('⚠️ Cluster still running. Terminate it with:')
    print(f'    emr.terminate_job_flows(JobFlowIds=["{cluster_id}"])')

# For a persistent cluster (KeepJobFlowAliveWhenNoSteps=True) you would run:
# emr.terminate_job_flows(JobFlowIds=[cluster_id])

print(f'\nS3 logs retained at: {logs_uri}/{cluster_id}/')
print('Delete them once you no longer need them for troubleshooting.')

---

## Summary

You prepared the bank marketing dataset with **EMR on EC2** and published feature-ready datasets to S3:

- `s3://<default-bucket>/bank-marketing-lab/data/train/train.csv`
- `s3://<default-bucket>/bank-marketing-lab/data/test/test.csv`

Because these paths are derived from the default bucket and the `bank-marketing-lab` prefix, **Lab 3A** picks them up automatically - no manual copy/paste needed. Continue to `lab3-model-build/lab3a_traditional_ml_experimenation.ipynb` to train the XGBoost model on this prepared data.

### Why this matters

- **Cluster-level control**: choose instance types, add bootstrap actions, install system packages, or attach the wider EMR ecosystem (Hive, Presto, Flink)
- **Transient by design**: `KeepJobFlowAliveWhenNoSteps=False` gives you that control without paying for an idle cluster
- **Same code, different substrate**: the PySpark job here is byte-identical to the one in Lab 2C — only the submission mechanics differ
- **Scalability**: handle larger data by adding core/task nodes, or by adding Spot task nodes for cost efficiency

### Choosing between the Lab 2 options

| Option | Compute | Startup | Best for |
|---|---|---|---|
| **2B** Processing Job | Managed container | ~3-4 min | Single-node Python/pandas prep; simplest path |
| **2C** EMR Serverless | Serverless Spark | ~1-2 min | Spark without capacity planning; bursty or unpredictable jobs |
| **2D** EMR on EC2 | Managed cluster | ~7-10 min | Long-running or repeated jobs; fine-grained cluster control |
| **2E** Canvas | Visual/no-code | Interactive | Exploration and prep without writing code |

At 41K rows this dataset fits comfortably in a single node, so 2B is the pragmatic choice — Spark's distributed execution costs more in startup than it saves in compute. The Spark options earn their keep once the data no longer fits on one machine, or when the same job must join several large sources.